# 05 — Queries and results: estimands, data ingress, reporting

The same observational table supports many scientific questions. This notebook
returns to the job-training extract and shows how to:

- ask interventional, ATE, and necessity/sufficiency queries;
- build a custom linear combination (experimental AST);
- ingest data from counts, records, or a DataFrame;
- use experimental closed-form, audit, and CI helpers;
- report a bound responsibly.


## Setup


In [1]:
from pathlib import Path

def tutorial_data(name: str) -> Path:
    """Resolve notebooks/data/<name> whether __file__ is set (CI) or not."""
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    candidates = [
        here / "data" / name,
        here / "notebooks" / "data" / name,
        Path.cwd() / "notebooks" / "data" / name,
        Path.cwd() / "data" / name,
    ]
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError(f"could not find notebooks/data/{name}; tried {candidates}")

import pandas as pd
from partialid import (
    Assumptions, Query, audit_bound, bound, bound_ci, from_counts,
    from_dataframe, from_records, try_closed_form,
)
from partialid.query_ast import Atom, LinComb, Lit, PO

df = pd.read_csv(tutorial_data("job_training.csv"))
data = from_dataframe(df, columns=["treated", "employed"])
a = Assumptions.order("treated", "employed") & Assumptions.mtr("employed", "treated")
print(f"N={len(df)}")


N=2000


## Built-in query faces

| Query | Plain-language ask |
|---|---|
| interventional | Employment rate if everyone were treated |
| ATE | Average gain from treating vs not |
| PNS | Probability treatment is necessary *and* sufficient for the outcome |
| PN / PS | Probabilities of necessity / sufficiency |


In [2]:
queries = {
    "interventional": Query.interventional(outcome="employed", do={"treated": 1}, value=1),
    "ATE": Query.ate(outcome="employed", treatment="treated"),
    "PNS": Query.pns(treatment="treated", outcome="employed"),
    "PN": Query.pn(treatment="treated", outcome="employed"),
    "PS": Query.ps(treatment="treated", outcome="employed"),
}
for name, Q in queries.items():
    r = bound(data.variables, data.P, Q, a)
    print(f"{name:14s}  {Q.describe()}")
    print(f"{'':14s}  [{r.lower:.4f}, {r.upper:.4f}]  w={r.width:.4f}  method={r.method}\n")


interventional  P(employed=1 | do(treated=1))
                [0.5575, 0.8575]  w=0.3000  method=fiber

ATE             ATE: P(employed=1 | do(treated=1)) - P(employed=1 | do(treated=0))
                [0.0000, 0.6535]  w=0.6535  method=fiber

PNS             PNS(treated→employed)
                [0.0000, 0.6535]  w=0.6535  method=fiber

PN              PN (probability of necessity)
                [0.0000, 1.0000]  w=1.0000  method=fiber

PS              PS (probability of sufficiency)
                [0.0000, 1.0000]  w=1.0000  method=fiber



## Custom linear combination (experimental)

The query AST (`PO`, `Lit`, `Atom`, `LinComb`) is **experimental**. Use named
factories when they suffice.


In [3]:
# 2 P(employed_1=1) - P(employed_0=1)
expr = LinComb((
    (2.0, Atom(Lit(PO("employed", {"treated": 1}), 1))),
    (-1.0, Atom(Lit(PO("employed", {"treated": 0}), 1))),
))
Q_custom = Query.from_expr(expr, outcome="employed")
r_custom = bound(data.variables, data.P, Q_custom, a)
print(Q_custom.describe())
print(r_custom.summary(Q_custom))


2*P(employed(treated=1)=1) - P(employed(treated=0)=1)
query: 2*P(employed(treated=1)=1) - P(employed(treated=0)=1)
interval: [0.5575, 1.511]
width: 0.9535
method: fiber
tight: True
success: True


## Conditional interventional

`condition=` restricts attention to a pretreatment slice of \(P\) before
bounding (see package docs for details).


In [4]:
# Illustrative: condition on treated==1 is *not* a pretreatment covariate —
# use a real covariate when you have one (notebook 06). Here we show the API
# shape on a trivial self-contained slice by rebuilding a tiny table with a
# binary covariate.
df_c = df.copy()
df_c["urban"] = (df_c.index % 2).astype(int)  # fake covariate for API demo only
data_c = from_dataframe(df_c, columns=["urban", "treated", "employed"])
Q_cond = Query.interventional(
    outcome="employed", do={"treated": 1}, value=1, condition={"urban": 1},
)
r_cond = bound(
    data_c.variables, data_c.P, Q_cond,
    Assumptions.order("urban", "treated", "employed"),
)
print(Q_cond.describe())
print(r_cond.summary(Q_cond))


P(employed=1 | do(treated=1)), urban=1
query: P(employed=1 | do(treated=1)), urban=1
interval: [0.344, 0.844]
width: 0.5
method: fiber
tight: True
success: True


## Data ingress cookbook


In [5]:
# 1) Aggregated counts
counts = {(0, 0): 30, (0, 1): 20, (1, 0): 15, (1, 1): 35}
d1 = from_counts(("treated", "employed"), counts)
# 2) Individual records
recs = [(0, 0), (0, 1), (1, 0), (1, 1), (1, 1)]
d2 = from_records(("treated", "employed"), recs)
# 3) DataFrame / CSV (already used above)
d3 = from_dataframe(df, columns=["treated", "employed"])

Q = Query.interventional(outcome="employed", do={"treated": 1}, value=1)
for label, d in [("counts", d1), ("records", d2), ("dataframe", d3)]:
    rr = bound(d.variables, d.P, Q, Assumptions.manski("treated", "employed"))
    print(f"{label:10s}  width={rr.width:.4f}  n_obs={d.n_obs}")


counts      width=0.5000  n_obs=None
records     width=0.4000  n_obs=None
dataframe   width=0.5040  n_obs=2000


## Closed form (experimental)

Some Manski / MTR settings admit analytic Moves. `try_closed_form` returns
them when available.


In [6]:
Q = Query.interventional(outcome="employed", do={"treated": 1}, value=1)
cf = try_closed_form(data.variables, data.P, Q, Assumptions.manski("treated", "employed"))
print(cf)


None


## Audit and confidence intervals (experimental)

`audit_bound` sanity-checks a result. `bound_ci` builds an Imbens–Manski-style
CI for the identified set (bootstrap; keep `n_boot` modest in notebooks).


In [7]:
Q = Query.interventional(outcome="employed", do={"treated": 1}, value=1)
r = bound(data.variables, data.P, Q, Assumptions.sensitivity_msm("treated", "employed", 3.0))
report = audit_bound(r, query=Q, assumptions=Assumptions.sensitivity_msm("treated", "employed", 3.0))
print(report)

ci = bound_ci(
    data.variables, data.P, Q,
    Assumptions.manski("treated", "employed"),
    n=data.n_obs,
    n_boot=40,
    seed=0,
)
print(ci)


AuditReport(query='P(employed=1 | do(treated=1))', method='msm_agg/msm_margin_binary_xy', tight=True, faces=[], budgets=['TanMSMAssumption'], interval=[0.4732,0.8092])
BoundCIResult(ci=[0.3355,0.8702], point=[0.3535,0.8575], level=0.95, method='imbens_manski')


## Reporting checklist

When you publish or hand off a bound, include:

1. **Sample** — N and how `P` was formed (counts / survey weights / …).
2. **Query** — `Q.describe()` in words.
3. **Assumptions** — full `a.describe()` list (not only a recipe name).
4. **Interval** — \([\ell, u]\), width, and `tight`.
5. **Sensitivity** — a path or breakdown if a budget is material (notebook 03).
6. **Failures** — if `success=False`, report infeasibility rather than inventing a point.

## Core path complete

Optional next steps:

- [06_panel_and_scale.ipynb](06_panel_and_scale.ipynb) — realistic 8-variable panel and scale.
- [07_advanced_identification.ipynb](07_advanced_identification.ipynb) — routing, degree, `prefer_poly`.
